# SiteSafe: PPE compliance with privacy
### Notebook 1 of 2: train, evaluate, export

**MAIB AI 221 Computer Vision with Applications, Project 6.**
Fine-tunes YOLO11n on Construction-PPE to detect helmets, vests, gloves, boots, goggles, people, forklifts and the
five "no_" classes (visibly missing PPE), then exports the model to ONNX for the browser app.

**Contents**
1. Configuration  2. Runtime check  3. Setup  4. Data  5. Training (with resume)  6. Evaluation on the held-out test split
7. Person-level compliance and calibration  8. ONNX export and int8 quantisation  9. YOLO11s trade-off  10. Results table

Everything is free tier: Colab T4, open data, open models. "Run all" works end to end, and can be re-run after a
disconnect: finished steps are skipped and training resumes from the last checkpoint on Drive.

In [ ]:
#@title 1. Configuration: every path and hyperparameter in one place
REPO_URL   = "https://github.com/Aftaab9/sitesafe.git"   # public repo with the scripts
DRIVE_DIR  = "/content/drive/MyDrive/sitesafe"            # checkpoints, plots and results survive a disconnect
SEED       = 42                                           # fixed in sitesafe/paths.py, repeated here for the record
MODEL      = "yolo11n.pt"
IMGSZ      = 640
EPOCHS     = 100
PATIENCE   = 20                                           # early stopping
BATCH      = 16
TRY_YOLO11S = True                                        # also train the larger model and report the trade-off

In [ ]:
#@title 2. Runtime check and time estimate
import subprocess, torch
assert torch.cuda.is_available(), "No GPU: Runtime > Change runtime type > T4 GPU"
print(torch.cuda.get_device_name(0))
print(subprocess.run(["nvidia-smi", "--query-gpu=memory.total,memory.used", "--format=csv"], capture_output=True, text=True).stdout)
print("Expected on a free T4: data 2 min, YOLO11n 100 epochs about 20-25 min, evaluation and export 3 min.")

In [ ]:
#@title 3. Mount Drive, get the code, install pinned requirements
from google.colab import drive
drive.mount("/content/drive")
import os
os.makedirs(DRIVE_DIR, exist_ok=True)
if not os.path.exists("/content/sitesafe"):
    !git clone -q {REPO_URL} /content/sitesafe
%cd /content/sitesafe
!pip -q install -r requirements-train.txt
# runs/ and results/ live on Drive so a disconnect loses nothing
for d in ("runs", "results", "models"):
    os.makedirs(f"{DRIVE_DIR}/{d}", exist_ok=True)
    if not os.path.islink(d):
        !rm -rf {d} && ln -s {DRIVE_DIR}/{d} {d}

## 4. Data

Construction-PPE (Ultralytics, **CC BY 4.0**): 800 train and 200 val images, 12 classes. The dataset has no test
split, so `make_split.py` carves a fixed, seeded, stratified 100-image **test** split out of train. Val is used
for model selection and thresholds only; test is touched once, for the final numbers.

In [ ]:
!python scripts/download_data.py
!python scripts/make_split.py
!python scripts/dataset_stats.py
from IPython.display import Image, display
display(Image("results/data/class_counts.png", width=760)); display(Image("results/data/samples_grid.jpg", width=900))

## 5. Training

Same seed, image size and schedule for every run so they are comparable. Each run appends one row
(configuration, metrics, minutes) to `results/runs.csv`. If the runtime disconnects, run this cell again:
it resumes from `runs/base/weights/last.pt`.

In [ ]:
import os
resume = "--resume" if os.path.exists("runs/base/weights/last.pt") and not os.path.exists("runs/base/results.png") else ""
if not os.path.exists("runs/base/results.png"):
    !python scripts/train.py --name base --model {MODEL} --epochs {EPOCHS} --patience {PATIENCE} --imgsz {IMGSZ} --batch {BATCH} --workers 2 --cache ram {resume}
display(Image("runs/base/results.png", width=900))

## 6. Evaluation on the held-out test split
Per-class precision, recall and AP, the confusion matrix and the precision-recall curves. The five `no_` classes are the ones that matter for safety and are the rarest.

In [ ]:
!python scripts/evaluate.py --weights runs/base/weights/best.pt --split test --tag base
import pandas as pd
display(pd.read_csv("results/eval/base_test.csv"))
for f in ("confusion_matrix_normalized.png", "BoxPR_curve.png", "val_batch0_pred.jpg"):
    p = f"results/eval/plots/base_test/{f}"
    if os.path.exists(p): display(Image(p, width=820))

## 7. Person-level compliance and calibration
PPE boxes are matched to people (intersection over PPE area plus a position check). Confidence is calibrated on val; per-class thresholds send low-confidence decisions to "check manually".

In [ ]:
!python scripts/export_onnx.py --weights runs/base/weights/best.pt --name sitesafe_base
!python scripts/calibrate.py
!python scripts/compliance_eval.py --tag base --split test
display(Image("results/eval/reliability.png", width=460))

## 8. Int8 quantisation
Static quantisation calibrated on training images; the accuracy cost is measured on the test split.

In [ ]:
!python scripts/quantize.py --name sitesafe_base
import json; print(json.dumps(json.load(open("results/eval/quantisation.json")), indent=2))

## 9. YOLO11s: is the bigger model worth it?
Same data and schedule. Reported: accuracy gain, file size and CPU time per image.

In [ ]:
if TRY_YOLO11S:
    if not os.path.exists("runs/base_s/results.png"):
        !python scripts/train.py --name base_s --model yolo11s.pt --epochs {EPOCHS} --patience {PATIENCE} --imgsz {IMGSZ} --batch {BATCH} --workers 2
    !python scripts/evaluate.py --weights runs/base_s/weights/best.pt --split test --tag base_s
    !python scripts/export_onnx.py --weights runs/base_s/weights/best.pt --name sitesafe_base_s

## 10. Results table
Every run so far. The files in `models/` and `results/` are on Drive; download `models/sitesafe_base.onnx` and `sitesafe_base.pt` into the repo's `models/` folder, then run `python scripts/build_web_assets.py`.

In [ ]:
display(pd.read_csv("results/runs.csv"))
!ls -la models/